# Fast titanium experiment inventory — raw item counts

Save this notebook in `aimdl_examples/tutorials`, select your Python environment, and **Run All**. It reads your existing `.env` or prompts privately for an API key. No earlier notebook is needed. Python 3.9+.

This version retrieves `xrd_raw` and `pdv_trace` metadata through `aimdl/datafiles`, intersects IGSNs locally with title-selected samples (including hyphenated children), and counts **distinct item IDs** by IGSN and experiment date. It never calls `item/{id}/files`, avoiding one request per matched item. No raw data bytes are downloaded.

**Interpretation:** `xrd_item_count` and `pdv_item_count` equal stored-file counts only if there is one file per item. They are not shot or experiment counts. Processed products and calibrants are excluded. Titles identify candidates, not all titanium-bearing samples. Child classification is inherited from the nearest matching ancestor and recorded explicitly.

Dates use the calendar date in `meta.experiment_date`, without timezone conversion or upload-date substitution. Missing and invalid dates remain visible. Results cover accessible items in the AIMD-L endpoint's scope; no server records are changed.

Validated locally with simulated API responses; authenticated execution takes place in your environment.


## 1. Dependencies
Run this optional cell only if imports in the next cell fail; uncomment its line. Then rerun the imports. No packages are installed automatically.


In [ ]:
# %pip install pandas girder-client python-dotenv


In [1]:
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

import pandas as pd
from dotenv import load_dotenv
from girder_client import GirderClient, HttpError
from IPython.display import display


## 2. Configuration and authentication
`Ti` is case-sensitive; “titanium” is case-insensitive. Review candidate titles for false positives. Add explicit IGSNs or exclusions below. Outputs go into a new timestamped directory each run. Only paginated metadata requests are made; progress is printed during scanning.


In [2]:
API_URL_OVERRIDE = None   # Optional explicit API URL
ENV_FILE = None           # Optional explicit path to your .env file
PAGE_SIZE = 100
INCLUDE_CHILDREN = True
TITLE_TERMS_CASE_SENSITIVE = ["Ti"]
TITLE_TERMS_CASE_INSENSITIVE = ["titanium"]
EXTRA_IGSNS = []          # Exact IGSNs to include even without a title match
EXCLUDE_IGSNS = []        # Exclude these IGSNs and their descendants
DATA_TYPES = {"xrd_raw": "XRD", "pdv_trace": "PDV"}
OUTPUT_BASE = Path.cwd() / "titanium_fast_inventory_outputs"

if ENV_FILE:
    env_path = Path(ENV_FILE).expanduser()
    if not env_path.is_file():
        raise FileNotFoundError(env_path)
else:
    env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / ".env").is_file()), None)
if env_path:
    load_dotenv(env_path, override=False)

api_url = (API_URL_OVERRIDE or os.getenv("GIRDER_API_URL")
           or "https://data.htmdec.org/api/v1").rstrip("/")
gc = GirderClient(apiUrl=api_url)
api_key = os.getenv("GIRDER_API_KEY") or getpass("Girder API key: ")
try:
    gc.authenticate(apiKey=api_key)
except Exception:
    raise RuntimeError("Authentication failed. Check the API URL and API key.") from None
finally:
    del api_key
user = gc.get("user/me")
print("Authenticated as:", user["login"])
print("API:", api_url)


Authenticated as: elbert
API: https://data.htmdec.org/api/v1


## 3. API pagination and matching helpers
No regex or other server-side metadata filters are used. HTTP failures stop the run rather than producing a silently incomplete report. Temporary HTTP 429/5xx failures receive a few short retries. Pagination continues to an empty page and detects repeated pages.


In [3]:
def api_get(endpoint, parameters=None):
    for attempt in range(4):
        try:
            return gc.get(endpoint, parameters=parameters)
        except HttpError as exc:
            status = getattr(exc, "status", None)
            if status in (429, 500, 502, 503, 504) and attempt < 3:
                time.sleep(2 ** attempt)
                continue
            raise


def api_pages(endpoint, parameters=None):
    offset, seen = 0, set()
    while True:
        batch = api_get(endpoint, {
            **(parameters or {}), "limit": PAGE_SIZE,
            "offset": offset, "sort": "_id", "sortdir": 1,
        })
        if not isinstance(batch, list):
            raise TypeError(f"{endpoint} returned {type(batch).__name__}, expected list")
        if not batch:
            return
        new_count = 0
        for record in batch:
            identifier = record["_id"]
            if identifier not in seen:
                seen.add(identifier)
                new_count += 1
                yield record
        if not new_count:
            raise RuntimeError(f"Repeated page from {endpoint}; pagination stopped")
        offset += len(batch)


def clean_igsn(value):
    return value.strip() if isinstance(value, str) else ""


def is_descendant(igsn, parent):
    return igsn == parent or (INCLUDE_CHILDREN and igsn.startswith(parent + "-"))


def excluded(igsn):
    return any(igsn == x or igsn.startswith(x + "-") for x in EXCLUDE_IGSNS)


def title_matches(title):
    return (any(term in title for term in TITLE_TERMS_CASE_SENSITIVE)
            or any(term.lower() in title.lower()
                   for term in TITLE_TERMS_CASE_INSENSITIVE))


def experiment_day(value):
    if value is None or str(value).strip() == "":
        return "UNKNOWN_DATE"
    try:
        return datetime.fromisoformat(str(value).strip().replace("Z", "+00:00")).date().isoformat()
    except (ValueError, TypeError):
        return "INVALID_DATE"


## 4. Retrieve and review candidate samples
The `samples` table is the title-matched (or explicitly included) list. Review it before continuing if you want to adjust the selection. To change it, edit configuration and rerun from configuration onward.


In [4]:
all_depositions = list(api_pages("deposition"))
lookup = {}
for record in all_depositions:
    igsn = clean_igsn(record.get("igsn"))
    if not igsn:
        continue
    titles = (record.get("metadata") or {}).get("titles") or []
    title = " | ".join(t.get("title", "") for t in titles if isinstance(t, dict))
    lookup[igsn] = {"igsn": igsn, "title": title, "deposition_id": record["_id"]}

missing_explicit = set(EXTRA_IGSNS) - set(lookup)
if missing_explicit:
    raise ValueError(f"Explicit IGSNs have no accessible deposition: {sorted(missing_explicit)}")
sample_rows = []
for igsn, record in lookup.items():
    if not excluded(igsn) and (title_matches(record["title"]) or igsn in EXTRA_IGSNS):
        sample_rows.append({**record, "selection_basis":
                            "explicit_igsn" if igsn in EXTRA_IGSNS else "title_match"})
samples = pd.DataFrame(sample_rows, columns=["igsn", "title", "deposition_id", "selection_basis"])
samples = samples.sort_values("igsn").reset_index(drop=True)
if samples.empty:
    raise ValueError("No candidate samples. Review title terms, permissions, or EXTRA_IGSNS.")
matched_ids = sorted(samples["igsn"], key=len, reverse=True)


selected_ids = set(matched_ids)
excluded_ids = set(EXCLUDE_IGSNS)
source_cache = {}


def title_source(igsn):
    if igsn in source_cache:
        return source_cache[igsn]
    # Walk exact ID and hyphen-delimited ancestors, most specific first.
    ancestors = [igsn]
    while "-" in ancestors[-1]:
        ancestors.append(ancestors[-1].rsplit("-", 1)[0])
    if any(x in excluded_ids for x in ancestors):
        result = None
    else:
        eligible = ancestors if INCLUDE_CHILDREN else ancestors[:1]
        result = next((x for x in eligible if x in selected_ids), None)
    source_cache[igsn] = result
    return result


print(f"{len(all_depositions):,} accessible depositions; {len(samples):,} candidates")
display(samples)


4,606 accessible depositions; 1,866 candidates


,igsn,title,deposition_id,selection_basis
0,APLMAL00003,Ti64 Etched Sample 1,6a6cb96fd64c1367b86d16b3,title_match
1,APLMAL00004,Ti64 Etched Sample 2,6a6cbae4c443dd8b646272dd,title_match
2,APLMAL00005,Ti64 Etched Sample 3,6a6cbc76c443dd8b646272e0,title_match
3,APLMAL00006,Titan23 Mech Polished,6a6cfc7fa7a7b6e805c0dc5f,title_match
4,APLMAL00006-001,Titan23 Mech Polished - 001,6a6cfc7fa7a7b6e805c0dc63,title_match
...,...,...,...,...
1861,NWXMAI00003,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c45531e775ae73d39d4a,title_match
1862,NWXMAI00004,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c4a7dd7c08f074d39d44,title_match
1863,NWXMAI00009,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c8a331e775ae73d39d4e,title_match
1864,NWXMAI00010,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807cb8c0f5e297109d39d88,title_match


## 5. Retrieve raw items and intersect IGSNs locally
Scan each requested data type once. Exact and ancestor IGSNs are checked against a set, with the result cached per observed IGSN. The API receives no regex filters. Diagnostics retain counts of items missing IGSNs. Pagination continues to an empty page, including when the server caps page sizes.


In [5]:
available_types = api_get("aimdl/datatype")
missing_types = set(DATA_TYPES) - set(available_types)
if missing_types:
    raise ValueError(f"Requested data types not available: {sorted(missing_types)}")
raw_items = {}
scan_rows = []
for data_type, technique in DATA_TYPES.items():
    scanned = matched = missing_igsn = 0
    for item in api_pages("aimdl/datafiles", {
        "dataType": data_type,
        "extraFields": json.dumps(["meta.igsn", "meta.experiment_date"]),
    }):
        scanned += 1
        igsn = clean_igsn((item.get("meta") or {}).get("igsn"))
        if not igsn:
            missing_igsn += 1
        elif title_source(igsn) is not None:
            raw_items[item["_id"]] = (technique, item)
            matched += 1
        if scanned % 1000 == 0:
            print(f"{data_type}: {scanned:,} scanned, {matched:,} matched", flush=True)
    scan_rows.append({"data_type": data_type, "scanned_items": scanned,
                      "matched_items": matched, "items_without_igsn": missing_igsn})
    print(f"{data_type}: complete — {scanned:,} scanned, {matched:,} matched", flush=True)
scan_diagnostics = pd.DataFrame(scan_rows)
display(scan_diagnostics)


xrd_raw: 1,000 scanned, 678 matched
xrd_raw: 2,000 scanned, 696 matched
xrd_raw: 3,000 scanned, 778 matched
xrd_raw: 4,000 scanned, 878 matched
xrd_raw: 5,000 scanned, 878 matched
xrd_raw: 6,000 scanned, 878 matched
xrd_raw: 7,000 scanned, 878 matched
xrd_raw: 8,000 scanned, 945 matched
xrd_raw: 9,000 scanned, 1,789 matched
xrd_raw: 10,000 scanned, 2,683 matched
xrd_raw: 11,000 scanned, 3,007 matched
xrd_raw: 12,000 scanned, 3,460 matched
xrd_raw: 13,000 scanned, 4,153 matched
xrd_raw: 14,000 scanned, 4,833 matched
xrd_raw: 15,000 scanned, 5,141 matched
xrd_raw: 16,000 scanned, 5,337 matched
xrd_raw: 17,000 scanned, 6,251 matched
xrd_raw: 18,000 scanned, 7,251 matched
xrd_raw: 19,000 scanned, 8,082 matched
xrd_raw: 20,000 scanned, 8,675 matched
xrd_raw: 21,000 scanned, 9,594 matched
xrd_raw: 22,000 scanned, 10,501 matched
xrd_raw: 23,000 scanned, 11,497 matched
xrd_raw: 24,000 scanned, 12,497 matched
xrd_raw: 25,000 scanned, 13,493 matched
xrd_raw: 26,000 scanned, 14,493 matched
xrd_ra

,data_type,scanned_items,matched_items,items_without_igsn
0,xrd_raw,49130,31556,0
1,pdv_trace,10937,7524,0


## 6. Build the item inventory — no additional API calls
The inventory contains one row per distinct matching item, its recorded date, technique, and the title responsible for its inclusion. Actual file counts are deliberately not estimated.


In [6]:
item_rows = []
for item_id, (technique, item) in raw_items.items():
    meta = item.get("meta") or {}
    igsn = clean_igsn(meta.get("igsn"))
    source = title_source(igsn)
    if source is None:
        raise RuntimeError("Selection changed; rerun from the candidate cell onward.")
    item_rows.append({
        "igsn": igsn,
        "title": lookup.get(igsn, {}).get("title", ""),
        "title_source_igsn": source,
        "matched_title": lookup[source]["title"],
        "match_relation": "exact" if source == igsn else "child_of_title_match",
        "experiment_date": experiment_day(meta.get("experiment_date")),
        "experiment_date_original": meta.get("experiment_date"),
        "technique": technique,
        "item_id": item_id,
        "item_name": item.get("name", ""),
    })
item_columns = ["igsn", "title", "title_source_igsn", "matched_title", "match_relation",
                "experiment_date", "experiment_date_original", "technique", "item_id", "item_name"]
items_inventory = pd.DataFrame(item_rows, columns=item_columns).drop_duplicates("item_id")
print(f"{len(items_inventory):,} distinct raw-data items; no per-item file requests made")
display(items_inventory.head(10))


39,080 distinct raw-data items; no per-item file requests made


,igsn,title,title_source_igsn,matched_title,match_relation,experiment_date,experiment_date_original,technique,item_id,item_name
0,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc642bc18c9c2ae35d42c5,scan_point_30_data_000001.h5
1,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6432c18c9c2ae35d42fe,scan_point_8_master.h5
2,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6434c18c9c2ae35d430a,scan_point_18_data_000001.h5
3,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6435c18c9c2ae35d430d,scan_point_27_data_000001.h5
4,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6435c18c9c2ae35d4313,scan_point_14_data_000001.h5
5,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6436c18c9c2ae35d4319,scan_point_16_data_000001.h5
6,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc6439c18c9c2ae35d431f,scan_point_8_data_000001.h5
7,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc643bc18c9c2ae35d4325,scan_point_12_data_000001.h5
8,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc643cc18c9c2ae35d432b,scan_point_6_data_000001.h5
9,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S3R3C1,Combinatorial synthesis of 200 micron thick gr...,exact,2025-09-29,2025-09-29T18:51:46+00:00,XRD,68dc643cc18c9c2ae35d432e,scan_point_17_data_000001.h5


## 7. Item-count summary and coverage
One row per actual IGSN and recorded experiment date, including `UNKNOWN_DATE` and `INVALID_DATE`. Zero means no retrieved raw items of that technique, not proof that no experiment occurred. Candidate coverage includes all selected descendants, even if they have their own matching titles. Candidates with no raw items remain in the coverage table without fabricated dates.


In [7]:
keys = ["igsn", "experiment_date"]
count_columns = ["xrd_item_count", "pdv_item_count"]
labels = ["igsn", "title", "title_source_igsn", "matched_title", "match_relation"]
if items_inventory.empty:
    summary = pd.DataFrame(columns=keys + labels[1:] + count_columns)
else:
    summary = (items_inventory.groupby(keys + ["technique"])["item_id"].nunique()
               .unstack("technique", fill_value=0)
               .reindex(columns=["XRD", "PDV"], fill_value=0)
               .rename(columns={"XRD": "xrd_item_count", "PDV": "pdv_item_count"})
               .rename_axis(columns=None).reset_index())
    summary = summary.merge(items_inventory[labels].drop_duplicates("igsn"),
                            on="igsn", how="left", validate="many_to_one")
    summary = summary[keys + labels[1:] + count_columns].sort_values(keys).reset_index(drop=True)

coverage_rows = []
for candidate in samples.to_dict("records"):
    mask = items_inventory["igsn"].map(lambda x: is_descendant(x, candidate["igsn"]))
    selected = items_inventory.loc[mask.astype(bool)]
    coverage_rows.append({**candidate, "matched_raw_items": len(selected),
                          "status": "no_raw_items" if selected.empty else "items_found"})
candidate_coverage = pd.DataFrame(coverage_rows)
date_issues = items_inventory.loc[items_inventory["experiment_date"].isin(["UNKNOWN_DATE", "INVALID_DATE"])]
assert int(summary[count_columns].sum().sum()) == len(items_inventory)
print(f"Summary rows: {len(summary):,}; items with date issues: {len(date_issues):,}")
display(summary)
print("Candidate coverage:")
display(candidate_coverage)
if not date_issues.empty:
    display(date_issues.head(20))


Summary rows: 534; items with date issues: 3


,igsn,experiment_date,title,title_source_igsn,matched_title,match_relation,xrd_item_count,pdv_item_count
0,APLMAL00003,2026-08-12,Ti64 Etched Sample 1,APLMAL00003,Ti64 Etched Sample 1,exact,100,60
1,APLMAL00006-001,2026-08-12,Titan23 Mech Polished - 001,APLMAL00006-001,Titan23 Mech Polished - 001,exact,50,0
2,APLMAL00006-001,2026-08-18,Titan23 Mech Polished - 001,APLMAL00006-001,Titan23 Mech Polished - 001,exact,100,75
3,APLMAL00006-002,2026-08-18,Titan23 Mech Polished - 002,APLMAL00006-002,Titan23 Mech Polished - 002,exact,100,72
4,APLMAL00007-004,2026-08-12,Titan23 Mech Polish - 004,APLMAL00007-004,Titan23 Mech Polish - 004,exact,50,0
...,...,...,...,...,...,...,...,...
529,JHXMAL00010,2026-10-02,TiVNb-20260608-841,JHXMAL00010,TiVNb-20260608-841,exact,100,0
530,JHXMAL00010,2026-10-15,TiVNb-20260608-841,JHXMAL00010,TiVNb-20260608-841,exact,50,0
531,NWXMAB00010-001-001,2026-07-03,Ti foil carburized - 001 - 001,NWXMAB00010-001-001,Ti foil carburized - 001 - 001,exact,170,0
532,NWXMAB00010-002-001,2026-07-03,Ti foil carburized - 002 - 001,NWXMAB00010-002-001,Ti foil carburized - 002 - 001,exact,170,0


Candidate coverage:


,igsn,title,deposition_id,selection_basis,matched_raw_items,status
0,APLMAL00003,Ti64 Etched Sample 1,6a6cb96fd64c1367b86d16b3,title_match,160,items_found
1,APLMAL00004,Ti64 Etched Sample 2,6a6cbae4c443dd8b646272dd,title_match,0,no_raw_items
2,APLMAL00005,Ti64 Etched Sample 3,6a6cbc76c443dd8b646272e0,title_match,0,no_raw_items
3,APLMAL00006,Titan23 Mech Polished,6a6cfc7fa7a7b6e805c0dc5f,title_match,397,items_found
4,APLMAL00006-001,Titan23 Mech Polished - 001,6a6cfc7fa7a7b6e805c0dc63,title_match,225,items_found
...,...,...,...,...,...,...
1861,NWXMAI00003,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c45531e775ae73d39d4a,title_match,0,no_raw_items
1862,NWXMAI00004,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c4a7dd7c08f074d39d44,title_match,0,no_raw_items
1863,NWXMAI00009,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807c8a331e775ae73d39d4e,title_match,0,no_raw_items
1864,NWXMAI00010,Hf25:Ti25:Ta25:Nb25 powder made via HEBM for 5 h.,6807cb8c0f5e297109d39d88,title_match,0,no_raw_items


,igsn,title,title_source_igsn,matched_title,match_relation,experiment_date,experiment_date_original,technique,item_id,item_name
3992,JHAMAC00001-S1R5C1,Combinatorial synthesis of 200 micron thick gr...,JHAMAC00001-S1R5C1,Combinatorial synthesis of 200 micron thick gr...,exact,UNKNOWN_DATE,None,XRD,699f938b7590bb7e1ed0ef93,scan_point_1_master.h5.missing_data
6306,JHAMAL00018-001,Grade 5 titanium - 001,JHAMAL00018-001,Grade 5 titanium - 001,exact,UNKNOWN_DATE,None,XRD,69dfe354917593d318ab0ecd,scan_point_18_master.h5
6773,JHAMAL00016-008,"Commercially pure, grade 2 titanium - 008",JHAMAL00016-008,"Commercially pure, grade 2 titanium - 008",exact,UNKNOWN_DATE,None,XRD,69dfe4d26cb2d97d89d763cf,scan_point_18_master.h5


## 8. Export CSVs and run settings
Save the summary, item inventory, title candidates, coverage, and diagnostics to a new local directory. Credentials are not exported. Counts reflect a live paginated inventory, not a transactionally frozen snapshot.


In [8]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = OUTPUT_BASE / run_stamp
output_dir.mkdir(parents=True, exist_ok=False)
exports = {
    "titanium_fast_item_summary.csv": summary,
    "titanium_items.csv": items_inventory,
    "titanium_candidates.csv": samples,
    "candidate_coverage.csv": candidate_coverage,
    "scan_diagnostics.csv": scan_diagnostics,
    "date_issues.csv": date_issues,
}
for filename, frame in exports.items():
    frame.to_csv(output_dir / filename, index=False)
settings = {
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "api_url": api_url,
    "include_children": INCLUDE_CHILDREN,
    "title_terms_case_sensitive": TITLE_TERMS_CASE_SENSITIVE,
    "title_terms_case_insensitive": TITLE_TERMS_CASE_INSENSITIVE,
    "extra_igsns": EXTRA_IGSNS, "exclude_igsns": EXCLUDE_IGSNS,
    "data_types": DATA_TYPES,
    "date_rule": "calendar date as recorded; no timezone conversion",
    "count_rule": "distinct raw-classified item IDs; not verified file counts",
    "summary_rows": len(summary), "raw_items": len(items_inventory),
}
(output_dir / "run_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Saved outputs to:", output_dir.resolve())


Saved outputs to: /Users/elbert/Documents/GitHub/openmsi/aimdl_examples/tutorials/titanium_fast_inventory_outputs/20261018T020600_627080Z


## Troubleshooting
- Missing packages: uncomment and run the optional installation cell.
- HTTP 401: check the key or restart and authenticate again. HTTP 403 indicates insufficient access; failures are not converted into zeros.
- No candidates: check title terms, permissions, and explicit inclusions. Alloy titles may omit titanium.
- No raw items: a sample may be outside this AIMD-L endpoint's scope, or data may lack an IGSN or classification.
- Missing dates: inspect `date_issues.csv`; no creation/upload dates are substituted.
- Changed selection: rerun from configuration onward to reset the matching cache.
- Use the original notebook when actual stored-file counts are required. This fast version counts items even if an item contains zero or multiple files.

References: [AIMD-L tutorial](https://github.com/openmsi/aimdl_examples/blob/main/tutorials/01_api_endpoints.ipynb), [API](https://data.htmdec.org/api/v1).
